# Sistema Experto - Peliculas/Series (CLIPS)

**Punto 1.** Recomienda que pelicula/serie ver segun el genero favorito, la compania, el tiempo disponible y el estado de animo del usuario.

## 1. Instalar e importar la libreria

In [1]:
!pip install clipspy
print("ok install")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 901.2/901.2 kB 18.2 MB/s eta 0:00:00
ok install


In [2]:
import clips
print("ok import")

ok import


## 2. Definicion del sistema experto (reglas CLIPS)

In [7]:
# Definición del sistema experto

REGLAS = [

    # 1. Filtro para familia
    """
    (defrule filtro_familiar
        (compania familia)
        =>
        (assert (filtro apto_todo_publico))
        (printout t "La compania es la familia: se filtran generos aptos para todo publico." crlf)
    )
    """,

    # 2. Advertencia: terror + familia
    """
    (defrule advertencia_terror_familia
        (filtro apto_todo_publico)
        (genero terror)
        =>
        (assert (advertencia "El genero terror no es recomendable para ver en familia."))
        (printout t "Advertencia: terror + familia." crlf)
    )
    """,

    # 3. Animo estresado
    """
    (defrule sugerencia_animo_estresado
        (animo estresado)
        =>
        (assert (sugerencia_genero comedia))
        (printout t "Si estas estresado, se sugiere una comedia para relajarte." crlf)
    )
    """,

    # 4. Poco tiempo
    """
    (defrule formato_poco_tiempo
        (tiempo poco)
        =>
        (assert (formato pelicula))
        (printout t "Hay poco tiempo disponible: se recomienda una pelicula." crlf)
    )
    """,

    # 5. Mucho tiempo + relajado
    """
    (defrule formato_maraton
        (tiempo mucho)
        (animo relajado)
        =>
        (assert (formato serie))
        (printout t "Hay mucho tiempo y el usuario esta relajado: se recomienda una serie." crlf)
    )
    """,

    # 6. Terror + solo
    """
    (defrule recomendar_terror_solo
        (genero terror)
        (compania solo)
        (not (recomendado))
        =>
        (assert (recomendacion "El Conjuro - ideal para una noche de terror en solitario."))
        (assert (recomendado))
    )
    """,

    # 7. Accion + amigos
    """
    (defrule recomendar_accion_amigos
        (genero accion)
        (compania amigos)
        (not (recomendado))
        =>
        (assert (recomendacion "Mad Max: Fury Road - accion y aventura para disfrutar con amigos."))
        (assert (recomendado))
    )
    """,

    # 8. Comedia + pareja
    """
    (defrule recomendar_comedia_pareja
        (genero comedia)
        (compania pareja)
        (not (recomendado))
        =>
        (assert (recomendacion "Loco por Mary - una comedia para disfrutar en pareja."))
        (assert (recomendado))
    )
    """,

    # 9. Ciencia ficcion + aventurero
    """
    (defrule recomendar_scifi_aventurero
        (genero ciencia_ficcion)
        (animo aventurero)
        (not (recomendado))
        =>
        (assert (recomendacion "Interestelar - ciencia ficcion y aventura para una experiencia intensa."))
        (assert (recomendado))
    )
    """,

    # 10. Drama + solo
    """
    (defrule recomendar_drama_solo
        (genero drama)
        (compania solo)
        (not (recomendado))
        =>
        (assert (recomendacion "Forrest Gump - una historia dramatica para disfrutar en solitario."))
        (assert (recomendado))
    )
    """,

    # 11. Recomendacion general de terror
    """
    (defrule recomendar_terror_general
        (genero terror)
        (not (recomendado))
        =>
        (assert (recomendacion "El Sexto Sentido - recomendacion general de terror y suspenso."))
        (assert (recomendado))
    )
    """,

    # 12. Recomendacion general de accion
    """
    (defrule recomendar_accion_general
        (genero accion)
        (not (recomendado))
        =>
        (assert (recomendacion "Mision Imposible - recomendacion general de accion y aventura."))
        (assert (recomendado))
    )
    """,

    # 13. Recomendacion general de comedia
    """
    (defrule recomendar_comedia_general
        (genero comedia)
        (not (recomendado))
        =>
        (assert (recomendacion "Intocable - recomendacion general de comedia y drama."))
        (assert (recomendado))
    )
    """,

    # 14. Recomendacion general de drama
    """
    (defrule recomendar_drama_general
        (genero drama)
        (not (recomendado))
        =>
        (assert (recomendacion "Forrest Gump - recomendacion general de drama."))
        (assert (recomendado))
    )
    """,

    # 15. Recomendacion general de ciencia ficcion
    """
    (defrule recomendar_scifi_general
        (genero ciencia_ficcion)
        (not (recomendado))
        =>
        (assert (recomendacion "Interestelar - recomendacion general de ciencia ficcion."))
        (assert (recomendado))
    )
    """
]

# Crear entorno CLIPS
env = clips.Environment()

# Cargar todas las reglas
for regla in REGLAS:
    env.build(regla)

print(f"OK - reglas cargadas: {len(REGLAS)}")

OK - reglas cargadas: 15


## 3. Prueba rapida por consola

In [4]:
# Prueba rapida por consola (igual que en clase: reset -> assert -> run -> revisar hechos)
env.reset()
env.assert_string("(genero terror)")
env.assert_string("(compania solo)")
env.assert_string("(tiempo poco)")
env.assert_string("(animo estresado)")
env.run()

for fact in env.facts():
    print(fact.template.name, "->", [fact[s] for s in fact.template.slots] if fact.template.slots else str(fact))

genero -> (genero terror)
compania -> (compania solo)
tiempo -> (tiempo poco)
animo -> (animo estresado)
sugerencia_genero -> (sugerencia_genero comedia)
formato -> (formato pelicula)
recomendacion -> (recomendacion "El Conjuro (terror) - ideal para verla solo/a")


## 4. Interfaz grafica de usuario (ipywidgets)

Elige los valores en los menus desplegables y presiona el boton.

In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output
print("ok import widgets")

ok import widgets


In [6]:
campos = {}
campos['genero'] = widgets.Dropdown(options=['accion', 'comedia', 'terror', 'drama', 'ciencia_ficcion'], value='accion', description='Genero favorito', style={'description_width': 'initial'})
campos['compania'] = widgets.Dropdown(options=['solo', 'pareja', 'amigos', 'familia'], value='solo', description='Compania', style={'description_width': 'initial'})
campos['tiempo'] = widgets.Dropdown(options=['poco', 'mucho'], value='poco', description='Tiempo disponible', style={'description_width': 'initial'})
campos['animo'] = widgets.Dropdown(options=['estresado', 'relajado', 'aventurero'], value='estresado', description='Estado de animo', style={'description_width': 'initial'})

boton = widgets.Button(description="Obtener recomendacion", button_style="success")
salida = widgets.Output()

def al_hacer_click(b):
    with salida:
        clear_output()
        env.reset()
        for nombre_hecho, widget in campos.items():
            env.assert_string(f"({nombre_hecho} {widget.value})")
        env.run()

        recomendaciones, advertencias, otros = [], [], []
        for fact in env.facts():
            nombre = fact.template.name
            if nombre == "recomendacion":
                recomendaciones.append(str(fact[0]))
            elif nombre == "advertencia":
                advertencias.append(str(fact[0]))
            elif nombre not in ("recomendacion", "advertencia"):
                otros.append(f"{nombre}: {fact[0]}")

        if otros:
            print("Hechos intermedios detectados:")
            for o in otros:
                print("  -", o)
            print()
        if advertencias:
            print("ADVERTENCIAS:")
            for a in advertencias:
                print("  -", a)
            print()
        if recomendaciones:
            print("RECOMENDACIONES:")
            for r in recomendaciones:
                print("  *", r)
        else:
            print("No se genero una recomendacion puntual con esta combinacion.")

boton.on_click(al_hacer_click)

display(widgets.VBox(list(campos.values()) + [boton, salida]))